In [2]:
import pandas as pd
import numpy as np

# Mount Google Drive so we can read the dataset straight from it.
from google.colab import drive
drive.mount('/content/drive')

# e.g. if it's in the top level of My Drive:
file_path = "/content/drive/MyDrive/kyleworker_1.csv"
df = pd.read_csv(file_path)

# The "Worker" column identifies which worker each row came from.
print(f"Loaded raw dataset: {df.shape}")
print(f"Workers present: {df['Worker'].unique().tolist()}")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded raw dataset: (13600, 8)
Workers present: ['worker 1']


,Worker,Timestamp,AccX,AccY,AccZ,GyroX,GyroY,GyroZ
0,worker 1,2026-07-27 13:06:50.046,0.26,-0.14,0.95,1.65,-2.87,-1.71
1,worker 1,2026-07-27 13:06:50.107,0.00,-0.10,0.97,10.62,-42.91,93.38
2,worker 1,2026-07-27 13:06:50.108,-0.09,0.09,0.91,10.38,-13.43,75.81
3,worker 1,2026-07-27 13:06:50.164,-0.09,0.08,0.94,-7.69,0.79,32.84
4,worker 1,2026-07-27 13:06:50.165,-0.09,0.08,0.94,-7.69,0.79,32.84


data cleaning, inspect structure

In [3]:
# Display basic information
print("--- Initial Overview ---")
print(f"Dataset Shape: {df.shape}")
print("\n--- Data Types & Missing Values ---")
print(df.info())
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

df.head()

--- Initial Overview ---
Dataset Shape: (13600, 8)

--- Data Types & Missing Values ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13600 entries, 0 to 13599
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Worker     13600 non-null  object 
 1   Timestamp  13600 non-null  object 
 2   AccX       13600 non-null  float64
 3   AccY       13600 non-null  float64
 4   AccZ       13600 non-null  float64
 5   GyroX      13600 non-null  float64
 6   GyroY      13600 non-null  float64
 7   GyroZ      13600 non-null  float64
dtypes: float64(6), object(2)
memory usage: 850.1+ KB
None

--- Missing Values Count ---
Worker       0
Timestamp    0
AccX         0
AccY         0
AccZ         0
GyroX        0
GyroY        0
GyroZ        0
dtype: int64


,Worker,Timestamp,AccX,AccY,AccZ,GyroX,GyroY,GyroZ
0,worker 1,2026-07-27 13:06:50.046,0.26,-0.14,0.95,1.65,-2.87,-1.71
1,worker 1,2026-07-27 13:06:50.107,0.00,-0.10,0.97,10.62,-42.91,93.38
2,worker 1,2026-07-27 13:06:50.108,-0.09,0.09,0.91,10.38,-13.43,75.81
3,worker 1,2026-07-27 13:06:50.164,-0.09,0.08,0.94,-7.69,0.79,32.84
4,worker 1,2026-07-27 13:06:50.165,-0.09,0.08,0.94,-7.69,0.79,32.84


In [4]:
# 1. Remove duplicate rows (ignoring pure identical duplicates)
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} duplicate rows.")

# 2. Handle missing values
# Forward-fill then back-fill sensor values (common for time-series IMU data)
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].ffill().bfill()

# Check remaining missing values
print(f"Remaining nulls: {df.isnull().sum().sum()}")

Removed 0 duplicate rows.
Remaining nulls: 0


In [5]:
# Parse the timestamp, then sort by Worker + Timestamp so each worker's
# series stays in chronological order (important before smoothing).
timestamp_col = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]

if timestamp_col:
    col_name = timestamp_col[0]
    df[col_name] = pd.to_datetime(df[col_name])
    df = df.sort_values(by=["Worker", col_name]).reset_index(drop=True)
    print(f"Parsed and sorted by ['Worker', '{col_name}']")
else:
    print("No timestamp column detected; skipping time sorting.")

Parsed and sorted by ['Worker', 'Timestamp']


In [6]:
# Smoothing = rolling mean, applied PER WORKER so values don't bleed across workers.
raw_sensor_cols = ["AccX", "AccY", "AccZ", "GyroX", "GyroY", "GyroZ"]

WINDOW = 5  # rolling window size; tune for more/less smoothing
for col in raw_sensor_cols:
    df[col + "_smooth"] = (
        df.groupby("Worker")[col]
          .transform(lambda s: s.rolling(window=WINDOW, min_periods=1).mean())
    )

print(f"Created smoothed columns with rolling window = {WINDOW}.")
df[[c + "_smooth" for c in raw_sensor_cols]].head()

Created smoothed columns with rolling window = 5.


,AccX_smooth,AccY_smooth,AccZ_smooth,GyroX_smooth,GyroY_smooth,GyroZ_smooth
0,0.260000,-0.1400,0.950000,1.650,-2.870000,-1.710000
1,0.130000,-0.1200,0.960000,6.135,-22.890000,45.835000
2,0.056667,-0.0500,0.943333,7.550,-19.736667,55.826667
3,0.020000,-0.0175,0.942500,3.740,-14.605000,50.080000
4,-0.002000,0.0020,0.942000,1.454,-11.526000,46.632000


In [7]:
# Clip extreme values with the IQR method on all numeric sensor columns
features = df.select_dtypes(include=[np.number]).columns

for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("Outlier clipping complete using IQR method.")

Outlier clipping complete using IQR method.


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Scale numeric feature columns (Worker/Timestamp are excluded automatically)
feature_cols = df.select_dtypes(include=[np.number]).columns

df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Features successfully scaled.")
df_scaled.head()

Features successfully scaled.


,Worker,Timestamp,AccX,AccY,AccZ,GyroX,GyroY,GyroZ,AccX_smooth,AccY_smooth,AccZ_smooth,GyroX_smooth,GyroY_smooth,GyroZ_smooth
0,worker 1,2026-07-27 13:06:50.046,0.975846,-0.165336,0.806664,0.028847,-0.040367,0.025254,0.994334,-0.167504,0.821344,0.032044,-0.047197,0.025574
1,worker 1,2026-07-27 13:06:50.107,0.511316,-0.084656,0.839043,0.240830,-0.861449,1.976805,0.756585,-0.125736,0.838048,0.152711,-0.508246,1.275321
2,worker 1,2026-07-27 13:06:50.108,0.350518,0.298571,0.741905,0.235158,-0.256916,1.802507,0.622470,0.020452,0.810209,0.190780,-0.435626,1.537958
3,worker 1,2026-07-27 13:06:50.164,0.350518,0.278401,0.790474,-0.191880,0.034687,0.817360,0.555412,0.088325,0.808817,0.088275,-0.317447,1.386904
4,worker 1,2026-07-27 13:06:50.165,0.350518,0.278401,0.790474,-0.191880,0.034687,0.817360,0.515178,0.129049,0.807982,0.026771,-0.246539,1.296271


In [9]:
output_cleaned_path = "kyle_master_preprocessed.csv"
df_scaled.to_csv(output_cleaned_path, index=False)

print(f"Preprocessed dataset successfully saved to: {output_cleaned_path}")
print(f"Final Processed Shape: {df_scaled.shape}")

Preprocessed dataset successfully saved to: kyle_master_preprocessed.csv
Final Processed Shape: (13600, 14)
